# IoT Sensor Simulation + CNN for Agroclimatic Detection

This notebook demonstrates a complete pipeline:

1. Simulation of IoT environmental sensors
2. Generation of synthetic image dataset (32x32 sensor maps)
3. Dataset labeling (normal vs thermal stress)
4. Training a Convolutional Neural Network (CNN)
5. Model evaluation

This structure is useful for research in **smart agriculture, environmental monitoring, and agroclimatic prediction**.

In [ ]:
# Install required libraries (Colab usually already has them)
!pip install numpy matplotlib tensorflow scikit-learn

## Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

## Simulate IoT Sensor Matrix

Each sensor reading is converted into a **32x32 matrix**, representing a spatial map of environmental measurements.

In [ ]:
def generate_sensor_matrix():
    base = np.random.normal(120, 40, (32,32))

    x = np.linspace(-1,1,32)
    y = np.linspace(-1,1,32)

    X, Y = np.meshgrid(x, y)

    pattern = np.exp(-(X**2 + Y**2)*3)*100

    sensor = base + pattern

    sensor = np.clip(sensor,0,255)

    return sensor

## Visualize a Sensor Map

In [ ]:
sample = generate_sensor_matrix()

plt.imshow(sample, cmap='viridis')
plt.colorbar()
plt.title('Simulated IoT Sensor Map')
plt.axis('off')
plt.show()

## Generate Dataset

In [ ]:
images = []
labels = []

for i in range(1000):

    img = generate_sensor_matrix()

    if img.mean() > 140:
        label = 1
    else:
        label = 0

    images.append(img)
    labels.append(label)

images = np.array(images)
labels = np.array(labels)

images = images / 255.0

images = images.reshape(-1,32,32,1)

## Train/Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    images,labels,test_size=0.2,random_state=42
)

## CNN Model

In [ ]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(32,32,1)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(64,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Train the Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test,y_test)
)

## Model Evaluation

In [ ]:
loss,acc = model.evaluate(X_test,y_test)

print('Test Accuracy:', acc)